In [3]:
!pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters langsmith langchain-groq langgraph langgraph-prebuilt

Found existing installation: langchain 1.2.12
Uninstalling langchain-1.2.12:
  Successfully uninstalled langchain-1.2.12
Found existing installation: langchain-core 1.2.19
Uninstalling langchain-core-1.2.19:
  Successfully uninstalled langchain-core-1.2.19
Found existing installation: langsmith 0.7.18
Uninstalling langsmith-0.7.18:
  Successfully uninstalled langsmith-0.7.18
Found existing installation: langgraph 1.1.2
Uninstalling langgraph-1.1.2:
  Successfully uninstalled langgraph-1.1.2
Found existing installation: langgraph-prebuilt 1.0.8
Uninstalling langgraph-prebuilt-1.0.8:
  Successfully uninstalled langgraph-prebuilt-1.0.8


In [4]:
!pip install -q langchain==0.2.16
!pip install -q langchain-core==0.2.43
!pip install -q langchain-community==0.2.16
!pip install -q langchain-text-splitters==0.2.4
!pip install -q langsmith==0.1.147
!pip install -q langchain-groq==0.1.5
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install gradio
!pip install -q markdown unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 require

In [1]:
import os
import math
import json
import pandas as pd

from langchain_groq import ChatGroq
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader, CSVLoader, JSONLoader, UnstructuredMarkdownLoader
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

from langchain.schema import Document
from google.colab import files

In [ ]:
os.environ["GROQ_API_KEY"] = ""

In [3]:
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.2
)

In [ ]:
# Fix: Address numpy dependency conflict and ensure sentence-transformers is installed.
# Explicitly installing numpy==1.26.4 to resolve conflicts with langchain and langchain-community.
!pip install --upgrade --force-reinstall numpy==1.26.4
!pip install --upgrade --force-reinstall sentence-transformers

In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_14124/2127729888.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
def compute_confidence(query, answer, context):

    # LLM self-eval
    llm_score_map = {"High": 0.9, "Medium": 0.6, "Low": 0.3}
    llm_conf = llm_confidence(query, answer, context)

    # Retrieval confidence
    docs_scores = vector_db.similarity_search_with_score(query, k=5)
    avg_score = sum(score for _, score in docs_scores) / len(docs_scores)
    retrieval_conf = 1 / (1 + avg_score)

    final_score = (llm_score_map[llm_conf] * 0.6) + (retrieval_conf * 0.4)

    if final_score > 0.7:
        return "High"
    elif final_score > 0.4:
        return "Medium"
    else:
        return "Low"

In [6]:
def llm_confidence(query, answer, context):

    prompt = f"""
    Evaluate the confidence of the following answer.

    Question: {query}
    Answer: {answer}

    Context:
    {context}

    Give a confidence score between 0 and 100.
    Only return the number.
    """

    response = llm.invoke(prompt)

    try:
        score = float(response.content.strip())
    except:
        score = 50

    if score >= 70:
        return "High"
    elif score >= 40:
        return "Medium"
    else:
        return "Low"

In [7]:
def classify_summary_type(query):

    prompt = f"""
    Determine the type of summary requested.

    Query: {query}

    Respond with ONLY one of the following labels:
    - document_summary
    - topic_summary

    If the user is asking to summarize a specific concept or subject, return topic_summary.
    If the user is asking to summarize the entire document(s), return document_summary.
    """

    response = llm.invoke(prompt)

    return response.content.strip().lower()

In [8]:
def summarize_document(query):

    summary_type = classify_summary_type(query)

    # TOPIC SUMMARY
    if summary_type == "topic_summary":

        retrieved_docs = retriever.get_relevant_documents(query)[:10]

        context = "\n".join([doc.page_content for doc in retrieved_docs])

        prompt = f"""
        Write a concise summary of the following topic using the context below.

        Topic Query:
        {query}

        Context:
        {context}
        """

        response = llm.invoke(prompt)

        confidence = compute_confidence(query,response,context)

        citations = list(set(
            doc.metadata.get("source","unknown")
            for doc in retrieved_docs
        ))

        return response.content, citations, confidence


    # DOCUMENT SUMMARY
    else:

        batch_size = 50
        chunk_summaries = []

        for i in range(0, len(documents), batch_size):

            batch = documents[i:i+batch_size:5]

            batch_text = "\n".join(
                doc.page_content for doc in batch
            )

            prompt = f"""
            Summarize the following section of a document:

            {batch_text}
            """

            response = llm.invoke(prompt)

            chunk_summaries.append(response.content)


        combined_summary = "\n".join(chunk_summaries)

        final_prompt = f"""
        The following are summaries of different parts of a document.

        {combined_summary}

        Write a concise overall summary of the entire document.
        """

        final_response = llm.invoke(final_prompt)

        confidence = compute_confidence(query,final_response,combined_summary)

        citations = list(set(
            doc.metadata.get("source","unknown")
            for doc in documents
        ))

        return final_response.content, citations, confidence

In [9]:
def qa_tool(query):

    retrieved_docs = retriever.get_relevant_documents(query)

    context = "\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
    Answer the question using ONLY the context below.

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.invoke(prompt)

    confidence = compute_confidence(query,response,context)

    citations = list(set([doc.metadata.get("source","unknown") for doc in retrieved_docs]))

    return response.content, citations, confidence

In [10]:
def analytical_tool(query):

    retrieved_docs = retriever.get_relevant_documents(query)

    context = "\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
    Analyze and compare the information across the following documents.

    Context:
    {context}

    Question:
    {query}

    Provide correlations, similarities, and differences if applicable.
    """

    response = llm.invoke(prompt)

    confidence = compute_confidence(query,response,context)

    citations = list(set([doc.metadata.get("source","unknown") for doc in retrieved_docs]))

    return response.content, citations, confidence

In [11]:
def agent_router(query):
    router_prompt = f"""
    Decide which tool should answer the user query.

    Tools:
    summarize_document → for summaries of documents or topics from various documents
    compare_document → for comparing or analysing or correlating multiple documents
    document_qa → for answering questions about one or multiple documents

    Query: {query}

    Return only the tool name.
    """

    tool_choice = llm.invoke(router_prompt).content.lower()

    if "summarize" in tool_choice:
        tool = "Summarizer Tool"
        answer, citations, confidence = summarize_document(query)

    elif "compare" in tool_choice:
        tool = "Comparative / Analytical Tool"
        answer, citations, confidence = analytical_tool(query)

    else:
        tool = "Q&A Tool"
        answer, citations, confidence = qa_tool(query)


    return {
        "Response": answer,
        "Tool Used": tool,
        "Citations": citations,
        "Confidence": confidence
    }

In [12]:
import gradio as gr
import os

# -------------------------------
# GLOBAL STORAGE
# -------------------------------
vector_db = None
documents = None


# -------------------------------
# FILE PROCESSING FUNCTION
# -------------------------------
def process_files(files):

    global vector_db, documents

    docs = []

    for file in files:
        file_path = file.name

        if file_path.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
            docs.extend(loader.load())

        elif file_path.endswith(".csv"):
            loader = CSVLoader(file_path)
            docs.extend(loader.load())

        elif file_path.endswith(".json"):
            loader = JSONLoader(file_path=file_path, jq_schema='.', text_content=False)
            docs.extend(loader.load())

        elif file_path.endswith(".md"):
            loader = UnstructuredMarkdownLoader(file_path)
            docs.extend(loader.load())

    # Chunking
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100
    )

    documents = text_splitter.split_documents(docs)

    # Create FAISS DB
    vector_db = FAISS.from_documents(documents, embedding_model)
    retriever = vector_db.as_retriever(search_kwargs={"k":10})


    return f"{len(files)} files processed | {len(documents)} chunks created"


# -------------------------------
# QUERY FUNCTION
# -------------------------------
def handle_query(query):

    if vector_db is None:
        return "Upload documents first!", "", "", ""

    result = agent_router(query)

    response = result["Response"]
    tool = result["Tool Used"]
    citations = "\n".join(result["Citations"])
    confidence = result["Confidence"]

    return response, tool, citations, confidence


# -------------------------------
# GRADIO UI
# -------------------------------
with gr.Blocks(title="Agentic RAG System") as demo:

    gr.Markdown("## 📄 Agentic RAG System")

    with gr.Row():
        file_input = gr.File(
            file_count="multiple",
            label="Upload Documents (PDF, CSV, JSON, Markdown)"
        )

    upload_btn = gr.Button("Process Documents")

    upload_status = gr.Textbox(label="Upload Status")

    upload_btn.click(
        fn=process_files,
        inputs=file_input,
        outputs=upload_status
    )

    gr.Markdown("### 🔍 Ask Questions")

    query_input = gr.Textbox(
        label="Enter your query",
        placeholder="Ask something about your documents..."
    )

    ask_btn = gr.Button("Run Query")

    # OUTPUT SECTION
    with gr.Row():
        response_box = gr.Textbox(label="Response", lines=10)
        tool_box = gr.Textbox(label="Tool Used")

    with gr.Row():
        citations_box = gr.Textbox(label="Citations", lines=5)
        confidence_box = gr.Textbox(label="Confidence")

    ask_btn.click(
        fn=handle_query,
        inputs=query_input,
        outputs=[
            response_box,
            tool_box,
            citations_box,
            confidence_box
        ]
    )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://485eb7bee2f5607e64.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://485eb7bee2f5607e64.gradio.live
